# Solution A3 · GridSearchCV from scratch

**Type**: Implement from scratch

## Objective

Implement a grid search with cross-validation and a final refit, and verify that it gives the same scores as `GridSearchCV`
on the same folds, including the `step__parameter` syntax.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

import itertools

from sklearn.base import clone
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = ds.loans(with_missing=False)
X = df[["age", "income", "job_tenure", "amount"]].to_numpy()
y = df["defaulted"].to_numpy()
estimator = make_pipeline(StandardScaler(), KNeighborsClassifier())
grid = {"kneighborsclassifier__n_neighbors": [5, 15, 30, 60], "kneighborsclassifier__weights": ["uniform", "distance"]}
folds = list(StratifiedKFold(5, shuffle=True, random_state=0).split(X, y))

## Commented solution

In [ ]:
def my_grid_search(estimator, grid, X, y, folds):
    keys = list(grid)
    rows = []
    for values in itertools.product(*grid.values()):
        params = dict(zip(keys, values))
        scores = []
        for tr, va in folds:
            m = clone(estimator).set_params(**params).fit(X[tr], y[tr])
            scores.append(roc_auc_score(y[va], m.predict_proba(X[va])[:, 1]))
        rows.append({**params, "mean_score": np.mean(scores)})
    results = pd.DataFrame(rows).sort_values("mean_score", ascending=False).reset_index(drop=True)
    best = {k: grid[k][grid[k].index(results.loc[0, k])] for k in keys}     # takes back the original value
    best_model = clone(estimator).set_params(**best).fit(X, y)              # refit on all the data
    return results, best, best_model

In [ ]:
results, best, best_model = my_grid_search(estimator, grid, X, y, folds)
sk = GridSearchCV(estimator, grid, cv=folds, scoring="roc_auc").fit(X, y)
n_trainings = len(results) * len(folds) + 1
print("best (mine):", best)
print("best (sk)  :", sk.best_params_)
print("total trainings:", n_trainings)
results.round(4)

## Check

In [ ]:
with verify():
    check(len(results) == 8 and "mean_score" in results.columns, "8 combinations with the mean score", "one row per combination")
    check(results["mean_score"].is_monotonic_decreasing, "results sorted from the best", "sort by mean_score descending")
    check(best == sk.best_params_, "the same best parameters as GridSearchCV", "best = dictionary of the parameters of the first row")
    check(close(np.sort(results["mean_score"].to_numpy()), np.sort(sk.cv_results_["mean_test_score"])),
          "the same mean scores as GridSearchCV", "mean of the AUC over the folds, for every combination")
    check(close(best_model.predict_proba(X), sk.best_estimator_.predict_proba(X)), "the refit matches best_estimator_",
          "the final model must be retrained on ALL the data with the best parameters")
    check(n_trainings == 8 * 5 + 1, "41 trainings: 8 combinations x 5 folds + the refit", "combinations x folds, plus one")
done()

## What you learned

- `GridSearchCV` is a double loop: for every combination, a complete cross-validation. The cost is combinations × folds + 1
  (the refit): with a grid of 5 parameters of 5 values and 5 folds that is 15,626 trainings.
- `cv_results_` is nothing more than the `results` table: the mean and the standard deviation across folds for every combination. Reading it
  in full (lesson 3.3 and exercise D1) is much more informative than the winner alone.
- `refit=True` retrains the best one on the **whole** training set: that model is the one to evaluate on the test set and put in production.

## Common pitfall

Reporting `best_score_` as the final estimate. It is the maximum of many estimates on the same folds, so it is biased upward: you measure it
in exercise C3.

## Interview question

<details><summary>How many trainings does GridSearchCV do with 3 hyperparameters of 4 values each and 5 folds?</summary>

4 × 4 × 4 = 64 combinations, for 5 folds: 320 trainings, plus one refit on the full training set: 321. It is the reason why with many
hyperparameters you move to RandomizedSearchCV or to methods that discard poor combinations early (halving).

</details>